# Prune rows from a dataset

Deletes a given list of rows from a CSV file and, optionally, the corresponding records from an associated JSONL file.

- The integers in `ROWS_TO_DELETE` are **0-based positional indices** into the data rows: index `0` is the first data row (the CSV header is **not** counted) and the first JSONL record.
- The CSV and JSONL are assumed to be row-aligned: data row *i* corresponds to JSONL record *i*.
- Outputs are written next to the inputs with `_pruned` appended to the stem (e.g. `data.csv` -> `data_pruned.csv`).

In [1]:
from pathlib import Path
import json
import csv
import sys

# Raise the default 128 KB CSV field limit (long generated text can exceed it).
_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(_limit)
        break
    except OverflowError:
        _limit //= 2

## Configuration

In [33]:
# Required: the CSV dataset.
CSV_PATH = Path("../out_data/final_results_v3_0615_results_vanilla_1.csv")

# Optional: an associated, row-aligned JSONL file. Set to None to skip.
# JSONL_PATH = None  # e.g. Path("../out_data/concatenated.jsonl")
JSONL_PATH = Path("../out_data/final_results_v3_0615_details_vanilla_1.jsonl")

# 0-based positional indices of the data rows / records to delete.
ROWS_TO_DELETE = [93, 94, 95, 151, 152, 154, 155, 156, 199, 200, 201, 202, 203, 207, 208, 215, 216, 217, 228, 229, 237, 241, 245]

In [34]:
def pruned_path(path: Path) -> Path:
    """Return ``path`` with ``_pruned`` appended to the stem, in the same directory."""
    return path.with_name(f"{path.stem}_pruned{path.suffix}")


# A set for O(1) membership tests; order/duplicates in the list don't matter.
delete_set = set(ROWS_TO_DELETE)
print(f"Deleting {len(delete_set)} row(s): {sorted(delete_set)}")

Deleting 23 row(s): [93, 94, 95, 151, 152, 154, 155, 156, 199, 200, 201, 202, 203, 207, 208, 215, 216, 217, 228, 229, 237, 241, 245]


## Prune the CSV

The header is always kept. Data rows are streamed and indexed from 0; any row whose index is in `delete_set` is dropped.

In [35]:
def prune_csv(path: Path, to_delete: set[int]) -> Path:
    """Write a copy of ``path`` with data rows at the given 0-based indices removed."""
    output = pruned_path(path)
    kept = deleted = 0

    with open(path, newline="", encoding="utf-8") as in_f, \
            open(output, "w", newline="", encoding="utf-8") as out_f:
        reader = csv.reader(in_f)
        writer = csv.writer(out_f)
        try:
            writer.writerow(next(reader))  # header
        except StopIteration:
            print(f"  {path.name} is empty")
            return output
        for index, row in enumerate(reader):
            if index in to_delete:
                deleted += 1
            else:
                writer.writerow(row)
                kept += 1

    print(f"CSV: kept {kept}, deleted {deleted} -> {output}")
    return output


csv_total = prune_csv(CSV_PATH, delete_set)

CSV: kept 399, deleted 23 -> ../out_data/final_results_v3_0615_results_vanilla_1_pruned.csv


## Prune the JSONL (optional)

Records are indexed from 0 (matching the CSV data rows). Skipped entirely when `JSONL_PATH` is `None`.

In [36]:
def prune_jsonl(path: Path, to_delete: set[int]) -> Path:
    """Write a copy of ``path`` with records at the given 0-based indices removed."""
    output = pruned_path(path)
    kept = deleted = 0

    with open(path, encoding="utf-8") as in_f, \
            open(output, "w", encoding="utf-8") as out_f:
        index = 0
        for line in in_f:
            if not line.strip():
                continue  # skip blank lines without consuming an index
            if index in to_delete:
                deleted += 1
            else:
                out_f.write(line if line.endswith("\n") else line + "\n")
                kept += 1
            index += 1

    print(f"JSONL: kept {kept}, deleted {deleted} -> {output}")
    return output


if JSONL_PATH is not None:
    prune_jsonl(JSONL_PATH, delete_set)
else:
    print("No JSONL_PATH set; skipping JSONL.")

JSONL: kept 399, deleted 23 -> ../out_data/final_results_v3_0615_details_vanilla_1_pruned.jsonl


In [37]:
JSONL_PATH

PosixPath('../out_data/final_results_v3_0615_details_vanilla_1.jsonl')